# 8/1 Revisions
 - Simulate durations from stickiness (kappa from gamma & rate)
 - Transitions rows sum to 1 with leftover spread uniformly
 - Include START and STOP tokens
 - Generate many random sentences for slow/fast; compute distributions
 - Apply distal local prior bump/penalty at decoding time to mimic rate expectation effect

In [ ]:
import numpy as np
from collections import Counter
from scipy.stats import norm, gamma as gamma_dist

np.random.seed(7)

In [ ]:
# ------------------------
# States and emissions
# ------------------------
STATES = ["START","lei","sure","or","time","STOP"]
EMIT_STATES = {"lei","sure","or","time"}
i = {s:k for k,s in enumerate(STATES)}

# 1D acoustic feature: 'lei' low, 'time' high, 'sure' and 'or' similar
mu = {
    "lei":  -1.0,
    "sure":  0.15,
    "or":    0.05,
    "time":  1.0
}
sigma = {s:0.45 for s in EMIT_STATES}  # std ~0.45

# ------------------------
# Base transitions (pi_bar) with uniform leftover mass
# ------------------------
def make_pi_bar(uniform_eps=0.0):
    n = len(STATES)
    T = np.zeros((n,n))
    def row_with_primary(from_s, primary):
        # primary: dict {to_state: weight}, the rest of mass is spread uniformly across *other* states
        row = np.zeros(n)
        used = 0.0
        for to_s, w in primary.items():
            row[i[to_s]] += w; used += w
        # spread leftover among all states NOT in primary
        leftover = max(0.0, 1.0 - used)
        mask = np.ones(n, dtype=bool)
        for to_s in primary.keys():
            mask[i[to_s]] = False
        # don't spread to START; do allow STOP to be in support except from STOP
        mask[i["START"]] = False
        if from_s == "STOP":
            mask[:] = False
            mask[i["STOP"]] = True
        count = mask.sum()
        if count > 0 and leftover > 0:
            row[mask] += leftover / count
        # small uniform epsilon everywhere (optional)
        if uniform_eps > 0:
            row += uniform_eps
        row = row / row.sum()
        return row

    # Primaries per mentor suggestion
    T[i["START"]] = row_with_primary("START", {"lei":0.97})
    T[i["lei"]]   = row_with_primary("lei",   {"sure":0.94})
    T[i["sure"]]  = row_with_primary("sure",  {"or":0.55, "time":0.40})
    T[i["or"]]    = row_with_primary("or",    {"time":0.95})
    T[i["time"]]  = row_with_primary("time",  {"STOP":0.96})
    T[i["STOP"]]  = row_with_primary("STOP",  {"STOP":1.0})
    return T

pi_bar = make_pi_bar(uniform_eps=0.0)



# ------------------------
# Stickiness mapping kappa(gamma, rate)
# ------------------------
def kappa_from_gamma_rate(gamma_val, rate, base=1.2, clip=(0.55,0.93)):
    # Slower (smaller rate) -> larger kappa; Faster -> smaller kappa
    val = 1.0 - np.exp(-base * (gamma_val + 0.25) / (rate + 1e-9))
    return float(np.clip(val, clip[0], clip[1]))

# draw per-state gamma
def sample_gammas(seed=None):
    rng = np.random.default_rng(seed)
    g = {}
    for s in STATES:
        if s in {"START","STOP"}:
            g[s] = 0.0
        else:
            # gamma(a=2,beta=5) mean 0.4
            g[s] = rng.gamma(shape=2.0, scale=1/5.0)
    return g

# Build full transition matrix at a given rate using stickiness + pi_bar
def build_T(rate, gammas):
    n = len(STATES)
    T = np.zeros((n,n))
    for s_from in STATES:
        row = pi_bar[i[s_from]].copy()
        if s_from in {"START","STOP"}:
            T[i[s_from]] = row
        else:
            kappa = kappa_from_gamma_rate(gammas[s_from], rate)
            row = (1 - kappa) * row
            row[i[s_from]] += kappa
            T[i[s_from]] = row / row.sum()
    return T

# Expected dwell in frames for a given state ~ geometric with self-prob kappa
def expected_dwell(kappa):
    return 1.0 / max(1e-6, (1.0 - kappa))


In [ ]:

# ------------------------
# Generative simulation
# ------------------------
def simulate_one(rate, gammas, max_frames=1000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    T = build_T(rate, gammas)
    s = "START"
    states = []
    emissions = []
    frame = 0
    while frame < max_frames:
        # transition
        probs = T[i[s]]
        s = rng.choice(STATES, p=probs)
        if s == "STOP":
            break
        states.append(s)
        # emission if needed
        if s in EMIT_STATES:
            x = rng.normal(loc=mu[s], scale=sigma[s])
            emissions.append(x)
        frame += 1
    return states, np.array(emissions).reshape(-1,1)




# ------------------------
# Time-local prior for decoding to emulate distal expectation bump
# ------------------------
def time_varying_logT(rate, gammas, T_len, t_star, window=4, bias_fast=0.9, bias_slow=-0.9):
    T_base = build_T(rate, gammas)
    # Make a list of log transition matrices
    logTs = []
    bias = bias_fast if rate >= 6.0 else bias_slow
    for t in range(T_len):
        Tt = T_base.copy()
        # local bump around expected 'or' onset: sure -> or
        if abs(t - t_star) <= window:
            Tt[i["sure"], i["or"]] *= np.exp(bias)
            # renormalize the row
            Tt[i["sure"]] /= Tt[i["sure"]].sum()
        logTs.append(np.log(Tt + 1e-12))
    return logTs

# expected t* from expected dwells (lei + half of sure)
def expected_tstar(rate, gammas):
    kap_lei  = kappa_from_gamma_rate(gammas["lei"],  rate)
    kap_sure = kappa_from_gamma_rate(gammas["sure"], rate)
    return int(round(expected_dwell(kap_lei) + 0.5 * expected_dwell(kap_sure)))





# ------------------------
# Viterbi with time-varying transitions
# ------------------------
def emission_log_probs(a_data):
    T = len(a_data)
    n = len(STATES)
    logp = np.full((T, n), -np.inf)
    for t in range(T):
        for s in EMIT_STATES:
            logp[t, i[s]] = norm.logpdf(a_data[t,0], loc=mu[s], scale=sigma[s])
    return logp

def viterbi_decode(a_data, logT_seq):
    T_len = len(a_data)
    n = len(STATES)
    V = np.full((T_len, n), -np.inf)
    B = np.zeros((T_len, n), dtype=int)
    logE = emission_log_probs(a_data)

    # Start: force START at virtual time -1, then first frame must leave START
    # So we initialize with log probs of transitioning out of START multiplied by emission
    # For simplicity, set initial to the columnwise transitions from START
    # But we need a state's emission too:
    start_row = logT_seq[0][i["START"]]  # log probs to all states
    # Only EMIT_STATES get emissions
    init = np.full(n, -np.inf)
    for sname in EMIT_STATES:
        init[i[sname]] = start_row[i[sname]] + logE[0, i[sname]]
    V[0] = init

    for t in range(1, T_len):
        prev = V[t-1][:, None] + logT_seq[t]  # n x n
        B[t] = np.argmax(prev, axis=0)
        V[t] = np.max(prev, axis=0) + logE[t]

    path = np.zeros(T_len, dtype=int)
    path[-1] = np.argmax(V[-1])
    for t in range(T_len-2, -1, -1):
        path[t] = B[t+1, path[t+1]]
    return [STATES[k] for k in path]

def compress(seq):
    out = []
    for s in seq:
        if not out or out[-1] != s:
            out.append(s)
    return out



In [ ]:
# ------------------------
# Monte Carlo simulations
# ------------------------
def run_experiment(n_trials=1000, rate_slow=4.0, rate_fast=7.0, use_local_bias=True, window=4, bias_fast=0.9, bias_slow=-0.9):
    rng = np.random.default_rng(1234)
    gammas = sample_gammas(seed=2025)  # fixed gammas for fair comparison across trials
    res_slow = []
    res_fast = []
    seq_slow = []
    seq_fast = []

    for _ in range(n_trials):
        # Slow: generate and decode with slow rate
        s_states, a = simulate_one(rate_slow, gammas, rng=rng)
        if len(a) < 3:
            continue
        if use_local_bias:
            tstar_slow = expected_tstar(rate_slow, gammas) + 4  # slight delay for slow
            logTs_slow = time_varying_logT(rate_slow, gammas, len(a), tstar_slow, window, bias_fast, bias_slow)
        else:
            base = build_T(rate_slow, gammas)
            logTs_slow = [np.log(base + 1e-12)] * len(a)
        zhat_slow = viterbi_decode(a, logTs_slow)
        seq_slow.append(compress(zhat_slow))

        # Fast: generate and decode with fast rate
        f_states, a2 = simulate_one(rate_fast, gammas, rng=rng)
        if len(a2) < 3:
            continue
        if use_local_bias:
            tstar_fast = expected_tstar(rate_fast, gammas)
            logTs_fast = time_varying_logT(rate_fast, gammas, len(a2), tstar_fast, window, bias_fast, bias_slow)
        else:
            base2 = build_T(rate_fast, gammas)
            logTs_fast = [np.log(base2 + 1e-12)] * len(a2)
        zhat_fast = viterbi_decode(a2, logTs_fast)
        seq_fast.append(compress(zhat_fast))

    def summarize(name, seqs):
        c = Counter(tuple(s) for s in seqs)
        total = sum(c.values())
        print(f"\n{name} (N={total}) top 8:")
        for (s, cnt) in c.most_common(8):
            print(f"  {list(s)!s:<40} | {cnt:4d} | {100*cnt/total:5.1f}%")
        has_or = sum(1 for s in seqs if "or" in s)
        print(f"  -> 'or' present: {has_or}/{total} = {100*has_or/total:4.1f}%")
        return c, total

    print("=== With local distal prior bump/penalty at decode ===" if use_local_bias else "=== No local bias (stickiness-only) ===")
    cslow, nslow = summarize(f"SLOW r={rate_slow}", seq_slow)
    cfast, nfast = summarize(f"FAST r={rate_fast}", seq_fast)
    return (cslow, nslow), (cfast, nfast)

# Run
res1 = run_experiment(n_trials=1200, use_local_bias=True, window=4, bias_fast=1.0, bias_slow=-1.0)
res2 = run_experiment(n_trials=1200, use_local_bias=False)


=== With local distal prior bump/penalty at decode ===

SLOW r=4.0 (N=1164) top 8:
  ['lei', 'sure', 'time']                  |  777 |  66.8%
  ['lei', 'sure']                          |  263 |  22.6%
  ['lei', 'sure', 'or', 'time']            |   82 |   7.0%
  ['lei', 'sure', 'lei', 'sure', 'time']   |    8 |   0.7%
  ['lei', 'time']                          |    5 |   0.4%
  ['lei', 'sure', 'lei', 'sure', 'or', 'time'] |    5 |   0.4%
  ['or', 'time']                           |    3 |   0.3%
  ['lei', 'sure', 'time', 'lei', 'sure']   |    3 |   0.3%
  -> 'or' present: 93/1164 =  8.0%

FAST r=7.0 (N=1131) top 8:
  ['lei', 'sure', 'or', 'time']            |  784 |  69.3%
  ['lei', 'sure', 'time']                  |  156 |  13.8%
  ['lei', 'sure', 'or']                    |   93 |   8.2%
  ['lei', 'sure']                          |   43 |   3.8%
  ['lei']                                  |    7 |   0.6%
  ['time']                                 |    6 |   0.5%
  ['lei', 'sure', 'or', 